In [0]:
metadata_definitions_df = spark.table("metadata_definitions")
metadata_definitions_df.show(truncate=False)


+------------+-----------------+-------------------------------------+
|table_name  |column_name      |description                          |
+------------+-----------------+-------------------------------------+
|weather_raw |station_id       |Unique identifier for weather station|
|weather_raw |city             |City where the station is located    |
|weather_raw |temperature_c    |Temperature measured in Celsius      |
|weather_raw |weather_condition|Observed weather condition           |
|station_info|station_id       |Unique identifier for weather station|
|station_info|region           |Geographical region code             |
|station_info|station_type     |Classification of weather station    |
+------------+-----------------+-------------------------------------+



In [0]:
actual_metadata_df = spark.table("actual_metadata")
metadata_definitions_df = spark.table("metadata_definitions")


In [0]:
joined_df = actual_metadata_df.join(
    metadata_definitions_df,
    on=["table_name", "column_name"],
    how="left"
)

joined_df.show(truncate=False)


+------------+-----------------+-------------------------------------+
|table_name  |column_name      |description                          |
+------------+-----------------+-------------------------------------+
|weather_raw |station_id       |Unique identifier for weather station|
|weather_raw |city             |City where the station is located    |
|weather_raw |temperature_c    |Temperature measured in Celsius      |
|weather_raw |weather_condition|Observed weather condition           |
|station_info|station_id       |Unique identifier for weather station|
|station_info|region           |Geographical region code             |
|station_info|station_type     |Classification of weather station    |
+------------+-----------------+-------------------------------------+



In [0]:
for row in joined_df.collect():
    if row.description is not None:
        spark.sql(f"""
            ALTER TABLE {row.table_name}
            ALTER COLUMN {row.column_name}
            COMMENT '{row.description}'
        """)


In [0]:
spark.sql("DESCRIBE EXTENDED weather_raw").show(truncate=False)


+----------------------------+--------------------------------------------------+-------------------------------------+
|col_name                    |data_type                                         |comment                              |
+----------------------------+--------------------------------------------------+-------------------------------------+
|station_id                  |bigint                                            |Unique identifier for weather station|
|city                        |string                                            |City where the station is located    |
|temperature_c               |double                                            |Temperature measured in Celsius      |
|weather_condition           |string                                            |Observed weather condition           |
|                            |                                                  |                                     |
|# Delta Statistics Columns  |          

In [0]:
spark.sql("DESCRIBE EXTENDED weather_raw").show(truncate=False)

+----------------------------+--------------------------------------------------+-------------------------------------+
|col_name                    |data_type                                         |comment                              |
+----------------------------+--------------------------------------------------+-------------------------------------+
|station_id                  |bigint                                            |Unique identifier for weather station|
|city                        |string                                            |City where the station is located    |
|temperature_c               |double                                            |Temperature measured in Celsius      |
|weather_condition           |string                                            |Observed weather condition           |
|humidity                    |double                                            |NULL                                 |
|                            |          

In [0]:
%sql
UPDATE metadata_definitions
SET description = 'City where weather station operates'
WHERE table_name = 'weather_raw'
AND column_name = 'city';


num_affected_rows
1


In [0]:
%sql
ALTER TABLE weather_raw ADD COLUMN humidity DOUBLE;
